# Dental AI — Kaggle 3D + TTA A/B testi

Bu notebook `work/vision48-3d-ready` dalındaki **son 3D arayüz kodunu** ve PyTorch `.pt` bulgu motorlarında **TTA (`augment=True`) açık/kapalı A/B karşılaştırmasını** test eder.

- Test edilen uygulama commit'i: `f5a700a75c9fd85cd2b3b68897a0cae63a4dc373`
- Production / `main` değiştirilmez.
- Liodon3 `.onnx` olduğu için bu TTA testine dahil edilmez.
- CLAHE production'da zorunlu ön işleme değildir; yalnız FDI motorunda sıfır tespit sonrası kurtarma/fallback olarak kalır.
- Panoramik tabanlı 3D görünüm CBCT veya gerçek medikal hacim değildir.

**Kaggle Internet açık olmalı. GPU T4 önerilir.** Kimliği belirlenebilir hasta verisi kullanmayın.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, sys, time
from collections import Counter

import requests
from IPython.display import HTML, display

REPOSITORY = 'https://github.com/dentalaidestek/dental-ai-v02.git'
BRANCH = 'work/vision48-3d-ready'
TEST_COMMIT = 'f5a700a75c9fd85cd2b3b68897a0cae63a4dc373'
WORKDIR = Path('/kaggle/working/dental-ai-v02')
print('Test commit:', TEST_COMMIT)


In [ ]:
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH], check=True)

subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--detach', TEST_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'], text=True).strip()
assert head == TEST_COMMIT, (head, TEST_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn[standard]', 'python-multipart',
    'ultralytics', 'huggingface_hub', 'scikit-image',
    'pandas', 'matplotlib', 'opencv-python-headless'
], check=True)
print('Kod ve bağımlılıklar hazır.')


In [ ]:
MODEL_DIR = WORKDIR / 'models' / 'vision'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assets = [
    ('vision-model-v1', 'YOLOv11x-seg.pt', '2d0c07b878e9f9730eb845a901e85abf3535d49cf4034d2246592f76201ac8af'),
    ('vision-findings9-v1', 'YOLO26_Dental_Findings_9.pt', '8070505857f354aae4f18bf621b9b79f0e3b48a2904e44f57a2bc598ed692849'),
    ('vision-impacted-v1', 'OralGuard_Impacted.pt', 'c3303656e72ede3f3d3229e58b2da276448fa204046d3e7a11e13f4d77bc723a'),
]

for tag, name, expected in assets:
    target = MODEL_DIR / name
    if not target.exists():
        url = f'https://github.com/dentalaidestek/dental-ai-v02/releases/download/{tag}/{name}'
        with requests.get(url, stream=True, timeout=300) as response:
            response.raise_for_status()
            with target.open('wb') as output:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        output.write(chunk)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    assert actual == expected, f'{name}: SHA256 uyuşmuyor'
    print(f'OK {name} ({target.stat().st_size/1024/1024:.1f} MB)')

# Liodon3 üretim 3D testinde kullanılabilsin diye indiriliyor;
# ancak ONNX olduğu için aşağıdaki augment=True A/B testine dahil edilmiyor.
LIODON_DIR = MODEL_DIR / 'optional' / 'liodon3'
LIODON_DIR.mkdir(parents=True, exist_ok=True)
liodon_target = LIODON_DIR / 'liodon_panorama3.onnx'
liodon_revision = '93c7037b11275d94cbf6c2f5d1ea86452910dc3a'
liodon_sha256 = '4cee38b54203634d895ed30a8910f5d7c4cefe22b18f9116b5561d9dd6e83a71'

if liodon_target.exists() and hashlib.sha256(liodon_target.read_bytes()).hexdigest() != liodon_sha256:
    liodon_target.unlink()

if not liodon_target.exists():
    url = f'https://huggingface.co/liodon-ai/dental-panoramic-detector/resolve/{liodon_revision}/best.onnx?download=true'
    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with liodon_target.open('wb') as output:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    output.write(chunk)

actual = hashlib.sha256(liodon_target.read_bytes()).hexdigest()
assert actual == liodon_sha256, 'Liodon3: SHA256 uyuşmuyor'
print(f'OK Liodon3 ({liodon_target.stat().st_size/1024/1024:.1f} MB)')


In [ ]:
checks = [
    ['node', '--check', 'vision_service/templates/viewer_v3.js'],
    ['node', '--check', 'vision_service/templates/viewer_v3_finalfix.js'],
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_3d*.py', '-v'],
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_root_filling_recovery.py', '-v'],
]

for command in checks:
    subprocess.run(command, cwd=WORKDIR, check=True)

engine_text = (WORKDIR / 'vision_service' / 'engine.py').read_text(encoding='utf-8')
assert 'cv2.createCLAHE' in engine_text
assert 'if not teeth:' in engine_text
assert 'clahe_sharp' in engine_text
print('3D kod kontrolleri geçti.')
print('CLAHE kontrolü: production FDI motorunda yalnız kurtarma/fallback akışında mevcut.')


## TTA A/B testi

Aşağıdaki hücre `/kaggle/input` altından en fazla **20 panoramik** bulur.  
İki `.pt` motor aynı görüntülerde aynı `imgsz/conf/iou` ayarlarıyla iki kez çalıştırılır:

- **TTA kapalı:** `augment=False`
- **TTA açık:** `augment=True`

Çıktıda süre, kutu sayısı, eşleşen tespitler, TTA ile eklenen/kaybolan adaylar ve ortalama confidence karşılaştırılır.  
Bu tablo tek başına “daha doğru” demek için yeterli değildir; yeni adayların gerçek/yanlış olduğu alttaki görsel karşılaştırmadan kontrol edilmelidir.


In [ ]:
import numpy as np
import pandas as pd
from ultralytics import YOLO

IMAGE_ROOT = Path('/kaggle/input')
MAX_IMAGES = 20
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

images = sorted(
    p for p in IMAGE_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in EXTS
)[:MAX_IMAGES]

if not images:
    raise RuntimeError('Kaggle Input bölümüne en az bir panoramik görüntü ekleyin.')

print(f'{len(images)} görüntü bulundu:')
for p in images:
    print(' -', p)


In [ ]:
TTA_MODELS = {
    'findings9': {
        'path': MODEL_DIR / 'YOLO26_Dental_Findings_9.pt',
        'imgsz': 1280,
        'conf': 0.35,
        'iou': 0.45,
    },
    'impacted_tooth': {
        'path': MODEL_DIR / 'OralGuard_Impacted.pt',
        'imgsz': 1280,
        'conf': 0.40,
        'iou': 0.45,
    },
}

def boxes_numpy(result):
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4)), np.empty((0,), dtype=int), np.empty((0,))
    return (
        result.boxes.xyxy.detach().cpu().numpy(),
        result.boxes.cls.detach().cpu().numpy().astype(int),
        result.boxes.conf.detach().cpu().numpy(),
    )

def iou_one(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0.0, x2-x1) * max(0.0, y2-y1)
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / max(1e-9, area_a + area_b - inter)

def match_detections(base_result, tta_result, threshold=0.50):
    b_boxes, b_cls, _ = boxes_numpy(base_result)
    t_boxes, t_cls, _ = boxes_numpy(tta_result)
    used = set()
    matched = 0
    for bb, bc in zip(b_boxes, b_cls):
        best_j = None
        best_iou = 0.0
        for j, (tb, tc) in enumerate(zip(t_boxes, t_cls)):
            if j in used or int(tc) != int(bc):
                continue
            ov = iou_one(bb, tb)
            if ov > best_iou:
                best_iou = ov
                best_j = j
        if best_j is not None and best_iou >= threshold:
            used.add(best_j)
            matched += 1
    return matched, len(t_boxes) - matched, len(b_boxes) - matched

models = {name: YOLO(str(cfg['path'])) for name, cfg in TTA_MODELS.items()}
rows = []
visual_cache = {}

for model_name, model in models.items():
    cfg = TTA_MODELS[model_name]
    for image_path in images:
        t0 = time.perf_counter()
        base = model.predict(
            source=str(image_path),
            imgsz=cfg['imgsz'],
            conf=cfg['conf'],
            iou=cfg['iou'],
            augment=False,
            verbose=False,
        )[0]
        base_s = time.perf_counter() - t0

        t0 = time.perf_counter()
        tta = model.predict(
            source=str(image_path),
            imgsz=cfg['imgsz'],
            conf=cfg['conf'],
            iou=cfg['iou'],
            augment=True,
            verbose=False,
        )[0]
        tta_s = time.perf_counter() - t0

        _, _, base_conf = boxes_numpy(base)
        _, _, tta_conf = boxes_numpy(tta)
        matched, added, lost = match_detections(base, tta, threshold=0.50)

        rows.append({
            'model': model_name,
            'image': image_path.name,
            'base_s': round(base_s, 3),
            'tta_s': round(tta_s, 3),
            'slowdown_x': round(tta_s / max(base_s, 1e-9), 2),
            'base_boxes': len(base_conf),
            'tta_boxes': len(tta_conf),
            'matched': matched,
            'added_by_tta': added,
            'lost_with_tta': lost,
            'base_mean_conf': round(float(base_conf.mean()), 4) if len(base_conf) else 0.0,
            'tta_mean_conf': round(float(tta_conf.mean()), 4) if len(tta_conf) else 0.0,
        })

        if len(visual_cache) < 4:
            visual_cache[(model_name, image_path.name)] = (base, tta)

df = pd.DataFrame(rows)
display(df)

summary = (
    df.groupby('model', as_index=False)
      .agg(
          images=('image', 'count'),
          base_s=('base_s', 'mean'),
          tta_s=('tta_s', 'mean'),
          slowdown_x=('slowdown_x', 'mean'),
          base_boxes=('base_boxes', 'sum'),
          tta_boxes=('tta_boxes', 'sum'),
          added_by_tta=('added_by_tta', 'sum'),
          lost_with_tta=('lost_with_tta', 'sum'),
          base_mean_conf=('base_mean_conf', 'mean'),
          tta_mean_conf=('tta_mean_conf', 'mean'),
      )
)
display(summary)

csv_path = Path('/kaggle/working/tta_ab_summary.csv')
df.to_csv(csv_path, index=False)
print('CSV kaydedildi:', csv_path)


In [ ]:
import matplotlib.pyplot as plt
import cv2

for (model_name, image_name), (base, tta) in visual_cache.items():
    base_img = cv2.cvtColor(base.plot(), cv2.COLOR_BGR2RGB)
    tta_img = cv2.cvtColor(tta.plot(), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(base_img)
    axes[0].set_title(f'{model_name} — TTA kapalı — {image_name}')
    axes[0].axis('off')
    axes[1].imshow(tta_img)
    axes[1].set_title(f'{model_name} — TTA açık — {image_name}')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

print('Karar kuralı: TTA yalnız gerçek bulguları artırıyor ve kabul edilebilir süre maliyeti getiriyorsa production motoruna alınmalı.')


## Son 3D arayüzü aç

Aşağıdaki hücreler test commit'indeki 3D sunucuyu açar. Böylece yeni:
- hayalî/ek atlas işaretlerinin kaldırıldığı görünümü,
- açık mavi transparan kemik + doğal beyaz diş + pembe kanal stilini,
- Bulgular / Tedavi / Görünüm panelini,
- bulguya dokununca geçici 3D vurgulamayı

aynı panoramik üzerinde kontrol edebilirsiniz.


In [ ]:
server_log_path = Path('/kaggle/working/dental_3d_server.log')
server_log = server_log_path.open('w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'vision_service.anatomy3d_direct_app:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=WORKDIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(server_log_path.read_text(errors='ignore')[-4000:])

print('Yerel 3D sunucusu hazır.')


In [ ]:
cloudflared = Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with cloudflared.open('wb') as output:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    output.write(chunk)
    cloudflared.chmod(0o755)

tunnel_log_path = Path('/kaggle/working/dental_3d_tunnel.log')
tunnel_log = tunnel_log_path.open('w')
tunnel = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(1)
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tunnel_log_path.read_text(errors='ignore'))
    if match:
        public_url = match.group(0) + '/viewer'
        break

if not public_url:
    raise RuntimeError(tunnel_log_path.read_text(errors='ignore')[-4000:])

display(HTML(
    f'<h3><a href="{public_url}" target="_blank">3D TEST EKRANINI AÇ</a></h3>'
    '<p>Link yalnız bu Kaggle oturumu çalışırken açıktır.</p>'
))
print(public_url)


## Kontrol listesi

1. Hayalî/çift kök veya eski renkli yuvarlak işaret kalmış mı?
2. Kemik açık mavi-transparan, dişler doğal beyaz ve kanal pembe mi?
3. Bulgular / Tedavi / Görünüm paneli 3D'nin hemen altında erişilebilir mi?
4. Bulgu kartına dokununca doğru FDI dişi geçici olarak vurgulanıyor mu?
5. Eksik diş ve gömülü diş yerleşimleri panoramiğe yaklaşık uyuyor mu?
6. TTA A/B görsellerinde yeni kutular gerçekten bulgu mu, yoksa yanlış pozitif mi?
7. TTA'nın süre maliyeti kabul edilebilir mi?

Panoramikten bukkolingual derinlik ölçülmediği için önden/arkadan derinlik hasta anatomisi olarak değerlendirilmemelidir.


In [ ]:
# Test bitince geçici bağlantıyı kapatın.
for process_name in ('tunnel', 'server'):
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
print('Geçici test sunucusu kapatıldı.')
